In [10]:
import polars as pl
from scipy.stats import norm
from plotly.offline import init_notebook_mode

init_notebook_mode(connected=True)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [11]:
budget = (
    pl.concat(
        [
            pl.read_json("budget2024.json", infer_schema_length=None),
            pl.read_json("budget2025.json", infer_schema_length=None),
        ]
    )
    .unnest("data")
    .unnest("budgetVsActualV2")
)

In [ ]:
def extract_cashflow(cashflow: pl.Series):
    return (
        cashflow.explode()
        .struct.unnest()
        .select("subAccounts", headline_category="category")
        .explode("subAccounts")
        .unnest("subAccounts")
        .select(
            "headline_category",
            "category",
            pl.col("annual").struct.field("transactions"),
        )
        .explode("transactions")
        .unnest("transactions")
        .with_columns(pl.col("date").str.to_date())
        .drop(["transactionType", "__typename"])
        .drop_nulls()
    )


expenses = extract_cashflow(budget["expenses"])

cashflow = pl.concat(
    [
        extract_cashflow(budget["incomes"]).with_columns(is_income=pl.lit(True)),
        expenses.with_columns(is_income=pl.lit(False)),
    ]
)

# Simulate 13 months because we need to see what happens this December to know how much budget we start next year with.
MONTHS_TO_SIMULATE = 13
SIMULATIONS = 10000
UNPAID_BILLS = 8828.77
# Based on the "Operation" account pulled from Daisy dashboard on 2025-11-18
STARTING_BALANCE = 24844.00 - UNPAID_BILLS
# Simulate budget increases between 1% - 5% in increments of 1 percentage point.
BUDGET_INCREASES = list(
    round(budget_increase / 100.0, 2) for budget_increase in range(-5, 7)
)
cashflow_by_month_by_increase = [
    (
        budget_increase,
        cashflow.with_columns(
            pl.col("amount")
            * pl.when(pl.col("is_income")).then(1 + budget_increase).otherwise(-1)
        )
        .group_by(
            year=pl.col("date").dt.year(),
            month=pl.col("date").dt.month(),
        )
        .agg(pl.col("amount").sum()),
    )
    for budget_increase in BUDGET_INCREASES
]

simulations = pl.concat(
    pl.collect_all(
        (
            cashflow_by_month.lazy()
            .select(
                pl.col("amount").sample(MONTHS_TO_SIMULATE, shuffle=True).cum_sum()
                + pl.lit(STARTING_BALANCE),
            )
            .with_columns(
                simulation_id=pl.lit(simulation_id),
                budget_increase=pl.lit(budget_increase),
            )
            .with_row_index("month")
            for (
                budget_increase,
                cashflow_by_month,
            ) in cashflow_by_month_by_increase
            for simulation_id in range(0, SIMULATIONS)
        )
    )
)

In [13]:
simulations_by_month = (
    simulations.group_by("month", "budget_increase")
    .agg(
        amount_average=pl.col("amount").mean(),
        amount_min=pl.col("amount").min(),
        amount_max=pl.col("amount").max(),
    )
    .sort("budget_increase", "month")
)

TRENDLINE_FIG_COL_COUNT = 2

fig = make_subplots(
    rows=int(len(BUDGET_INCREASES) / TRENDLINE_FIG_COL_COUNT),
    cols=TRENDLINE_FIG_COL_COUNT,
    shared_yaxes="all",
    subplot_titles=[
        f"budget_increase={budget_increase}" for budget_increase in BUDGET_INCREASES
    ],
)
for i, budget_increase in enumerate(BUDGET_INCREASES):
    sims = simulations_by_month.filter(pl.col("budget_increase") == budget_increase)
    row = int(i / TRENDLINE_FIG_COL_COUNT) + 1
    col = i % TRENDLINE_FIG_COL_COUNT + 1
    fig.add_trace(
        go.Scatter(
            x=pl.concat([sims["month"], sims["month"].reverse()]),
            y=pl.concat([sims["amount_min"], sims["amount_max"].reverse()]),
            name=f"min/max {budget_increase}",
            fill="toself",
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(x=sims["month"], y=sims["amount_average"], name=budget_increase),
        row=row,
        col=col,
    )
    fig.update_xaxes(title_text="month", row=row, col=col)
fig.update_layout(height=1000, legend=go.layout.Legend(title="budget_increase"))
fig.show()

In [14]:
simulation_mins = simulations.group_by("budget_increase", "simulation_id").agg(
    pl.col("amount").min()
)

In [15]:
fig = px.pie(
    simulation_mins.group_by(
        "budget_increase",
        ruinous=pl.col("amount") < 0,
    )
    .len("simulation_count")
    .with_columns(
        ruinous=pl.when("ruinous")
        .then(pl.lit("Special Assessment"))
        .otherwise(pl.lit("Safe"))
    )
    .sort("budget_increase"),
    names="ruinous",
    values="simulation_count",
    facet_col="budget_increase",
    facet_col_wrap=3,
    title="Likelihood of Special Assessment",
    color_discrete_sequence=["#4B08AF", "#32965D"],
    height=1000,
)
fig.show(renderer="notebook_connected")

# Special Assessment
95% confidence that the special assessment--if there is one--will be less than `amount`

In [16]:
# Inflation estimation from https://www.federalreserve.gov/monetarypolicy/files/fomcprojtabl20250917.pdf
INFLATION = 0.026
simulation_mins.filter(pl.col("amount") < 0).sort("budget_increase").with_columns(
    -pl.col("amount")
).group_by("budget_increase").agg(
    amount_95_conf=pl.col("amount").mean() + pl.col("amount").std() * norm.ppf(0.95),
    amount_99_conf=pl.col("amount").mean() + pl.col("amount").std() * norm.ppf(0.99),
).with_columns(
    amount_95_conf_with_inflation=pl.col("amount_95_conf") * (1 + INFLATION),
    amount_99_conf_with_inflation=pl.col("amount_99_conf") * (1 + INFLATION),
)

budget_increase,amount_95_conf,amount_99_conf,amount_95_conf_with_inflation,amount_99_conf_with_inflation
f64,f64,f64,f64,f64
-0.05,32022.855033,38981.015593,32855.449264,39994.521998
-0.04,30502.832122,37219.707625,31295.905757,38187.420023
-0.03,28964.935197,35436.985629,29718.023512,36358.347256
-0.02,27499.827383,33672.050227,28214.822895,34547.523533
-0.01,25948.529556,31831.221753,26623.191324,32658.833519
…,…,…,…,…
0.02,21755.591775,26780.372621,22321.237161,27476.662309
0.03,21068.784555,25847.620959,21616.572953,26519.659104
0.04,20121.784243,24739.457731,20644.950633,25382.683632


In [17]:
fig = px.bar(
    expenses.with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="headline_category",
)
fig.show(renderer="notebook_connected")

In [18]:
fig = px.bar(
    expenses.with_columns(
        chart_date=pl.date(pl.col("date").dt.year(), pl.col("date").dt.month(), 1)
    ),
    x="chart_date",
    y="amount",
    color="category",
)
fig.show(renderer="notebook_connected")